In [0]:
dbutils.widgets.text("catalog", "dev")
catalog = dbutils.widgets.get("catalog")

from pyspark.sql import functions as F, Window

In [0]:
region_std = (
    spark.read.table(f"{catalog}.bronze.region")
    .select(
        "r_regionkey",
        F.upper(F.trim(F.col("r_name"))).alias("r_name"),
        F.trim(F.col("r_comment")).alias("r_comment"),
        "_ingested_at",
        "_source_file",
    )
)

pk_col = "r_regionkey"

w = Window.partitionBy(pk_col)

region_checked = (
    region_std
    .withColumn("_pk_count", F.count(F.lit(1)).over(w))
    .withColumn(
        "quarantine_reason",
        F.when(F.col(pk_col).isNull(), F.lit("missing_primary_key"))
        .when(
            F.col("_pk_count") > 1, "duplicate primary key"
        )
    )
)

region_valid = (
    region_checked
    .filter(F.col("quarantine_reason").isNull())
    .drop("_pk_count", "quarantine_reason", "_ingested_at", "_source_file")
)


region_quarantined = (
    region_checked
    .filter(F.col("quarantine_reason").isNotNull())
    .select(
        "quarantine_reason",
        F.to_csv(F.struct(*region_std.columns)).alias("raw_data"),
    )
)

print(f"region: valid = {region_valid.count()}, quarantined = {region_quarantined.count()}")


region_valid.write.mode("overwrite").saveAsTable(f"{catalog}.silver.region")

# Quarantine table: raw row as a CSV string + reason (no volume, no external files)
region_quarantined.write.mode("overwrite").saveAsTable(f"{catalog}.silver.region_quarantine")

display(region_quarantined)

In [0]:
nation_std = (
    spark.read.table(f"{catalog}.bronze.nation")
    .select(
        "n_nationkey",
        F.upper(F.trim(F.col("n_name"))).alias("n_name"),
        "n_regionkey",
        F.trim(F.col("n_comment")).alias("n_comment"),
        "_ingested_at",
        "_source_file",
    )
)

pk_col = "n_nationkey"

w = Window.partitionBy(pk_col)

valid_region_keys = spark.read.table(f"{catalog}.silver.region").select(F.col("r_regionkey").alias("_valid_region_key"))

nation_checked = (
    nation_std
    .join(valid_region_keys, F.col("n_regionkey") == F.col("_valid_region_key"), how="inner")
    .withColumn("_pk_count", F.count(F.lit(1)).over(w))
    .withColumn(
        "quarantine_reason",
        F.array_compact(
            F.array(
                F.when(F.col(pk_col).isNull(), F.lit("missing_primary_key")),
                F.when(
                    F.col("_pk_count") > 1, "DUPLICATE PRIMARY KEY"
                ),
                F.when(F.col("n_regionkey").isNull() | F.col("_valid_region_key").isNull(), "INVALID REGION KEY")
            )
        )
    )
)

nation_valid = (
    nation_checked.filter(F.size(F.col("quarantine_reason")) == 0)
    .drop("_pk_count", "quarantine_reason", "_ingested_at", "_source_file", "_valid_region_key")
)


nation_quarantined = (
    nation_checked
    .filter(F.size(F.col("quarantine_reason")) > 0)
    .select(
        F.to_csv(F.struct(*nation_std.columns)).alias("raw_data"),
        F.array_join(F.col("quarantine_reason"), ", ").alias("quarantine_reason"),
    )
)

print(f"nation: valid = {nation_valid.count()}, quarantined = {nation_quarantined.count()}")


nation_valid.write.mode("overwrite").saveAsTable(f"{catalog}.silver.nation")

nation_quarantined.write.mode("overwrite").saveAsTable(f"{catalog}.silver.nation_quarantine")

display(nation_quarantined)

In [0]:
customer_std = (
    spark.read.table(f"{catalog}.bronze.customer")
    .select(
        "c_custkey",
        F.trim(F.col("c_name")).alias("c_name"),
        "c_address",
        "c_nationkey",
        "c_phone",
        "c_acctbal",
        F.upper(F.trim(F.col("c_mktsegment"))).alias("c_mktsegment"),
        F.trim(F.col("c_comment")).alias("c_comment"),
        "_ingested_at",
        "_source_file",
    )
)

pk_col = "c_custkey"

valid_nation_keys = spark.read.table(f"{catalog}.silver.nation").select(
    F.col("n_nationkey").alias("_valid_nation_key")
)

customer_validated = (
    customer_std
    .join(valid_nation_keys, F.col("c_nationkey") == F.col("_valid_nation_key"), how="left")
    .withColumn(
        "quarantine_reason",
        F.array_compact(
            F.array(
                F.when(F.col(pk_col).isNull(), "INVALID PRIMARY KEY"),
                F.when(F.col("c_nationkey").isNull(), "NO NATION KEY PRESENT"),
                F.when(F.col("_valid_nation_key").isNull(), "INVALID NATION KEY")
            )
        )
    )
)



w = Window.partitionBy(pk_col).orderBy(F.col("_ingested_at").desc())

customer_deduped = (
    customer_validated
    .filter(F.size(F.col("quarantine_reason")) == 0)
    .withColumn(
        "quarantine_reason",
        F.when(
            F.row_number().over(w) > 1 , "DUPLICAtE PRIMARY KEY"
        ),
    )
)

customer_valid = (
    customer_deduped
    .filter(F.col("quarantine_reason").isNull())
    .drop("quarantine_reason", "_valid_nation_key", "_ingested_at", "_source_file")
)

customer_quarantined = (
    customer_validated
    .filter(F.size(F.col("quarantine_reason")) > 0)
    .select(
        F.to_csv(F.struct(*customer_std.columns)).alias("raw_data"),
        F.array_join(F.col("quarantine_reason"), ", ").alias("quarantine_reason"),
    )
    .unionByName(
        customer_deduped
        .filter(F.col("quarantine_reason").isNotNull())
        .select(
            F.to_csv(F.struct(*customer_std.columns)).alias("raw_data"),
            "quarantine_reason",
        )
    )
)

print(f"customer: valid = {customer_valid.count()}, quarantined = {customer_quarantined.count()}")

customer_valid.write.mode("overwrite").saveAsTable(f"{catalog}.silver.customer")
customer_quarantined.write.mode("overwrite").saveAsTable(f"{catalog}.silver.customer_quarantine")

display(customer_quarantined)

In [0]:
orders_std = (
    spark.read.table(f"{catalog}.bronze.orders")
    .select(
        "o_orderkey",
        "o_custkey",
        F.upper(F.trim(F.col("o_orderstatus"))).alias("o_orderstatus"),
        "o_totalprice",
        "o_orderdate",
        "o_orderpriority",
        "o_clerk",
        "o_shippriority",
        "o_comment",
        "_ingested_at",
        "_source_file",
    )
)

pk_col = "o_orderkey"


valid_customer_keys = spark.read.table(f"{catalog}.silver.customer").select(
    F.col("c_custkey").alias("_valid_customer_key")
)

orders_validated = (
    orders_std
    .join(valid_customer_keys, F.col("o_custkey") == F.col("_valid_customer_key"), how="left")
    .withColumn(
        "quarantine_reason",
        F.array_compact(
            F.array(
                F.when(F.col(pk_col).isNull(), "INVALID PRIMARY KEY"),
                F.when(F.col("o_custkey").isNull(), "NO CUSTOMER KEY PRESENT"),
                F.when(F.col("_valid_customer_key").isNull(), "INVALID NATION KEY"),
                F.when(F.col("o_totalprice") <= 0, "NON POSITIVE TOTALPRICE"),
                F.when(F.col("o_orderdate") < "1992-01-01", "ORDER DATE BEFORE 1992"),
                F.when(F.col("o_orderdate") > "1998-12-31", "ORDER DATE AFTER 1998"),
            )
        )
    )
)



w = Window.partitionBy(pk_col).orderBy(F.col("_ingested_at").desc())

orders_deduped = (
    orders_validated
    .filter(F.size(F.col("quarantine_reason")) == 0)
    .withColumn(
        "quarantine_reason",
        F.when(
            F.row_number().over(w) > 1 , "DUPLICAtE PRIMARY KEY"
        ),
    )
)

orders_valid = (
    orders_deduped
    .filter(F.col("quarantine_reason").isNull())
    .drop("quarantine_reason", "_valid_nation_key", "_ingested_at", "_source_file")
)

orders_quarantined = (
    orders_validated
    .filter(F.size(F.col("quarantine_reason")) > 0)
    .select(
        F.to_csv(F.struct(*orders_std.columns)).alias("raw_data"),
        F.array_join(F.col("quarantine_reason"), ", ").alias("quarantine_reason"),
    )
    .unionByName(
        orders_deduped
        .filter(F.col("quarantine_reason").isNotNull())
        .select(
            F.to_csv(F.struct(*orders_std.columns)).alias("raw_data"),
            "quarantine_reason",
        )
    )
)

print(f"orders: valid = {orders_valid.count()}, quarantined = {orders_quarantined.count()}")

orders_valid.write.mode("overwrite").saveAsTable(f"{catalog}.silver.order")
orders_quarantined.write.mode("overwrite").saveAsTable(f"{catalog}.silver.order_quarantine")

display(orders_quarantined)

In [0]:
supplier_std = (
    spark.read.table(f"{catalog}.bronze.supplier")
    .select(
        "s_suppkey",
        F.trim(F.col("s_name")).alias("s_name"),
        F.trim(F.col("s_address")).alias("s_address"),
        "s_nationkey",
        F.trim(F.col("s_phone")).alias("s_phone"),
        "s_acctbal",
        F.trim(F.col("s_comment")).alias("s_comment"),
        "_ingested_at",
        "_source_file",
    )
)

pk_col = "s_suppkey"

valid_nation_keys = spark.read.table(f"{catalog}.silver.nation").select(
    F.col("n_nationkey").alias("_valid_nation_key")
)

supplier_validated = (
    supplier_std
    .join(valid_nation_keys, F.col("s_nationkey") == F.col("_valid_nation_key"), how="left")
    .withColumn(
        "quarantine_reason",
        F.array_compact(
            F.array(
                F.when(F.col(pk_col).isNull(), "INVALID PRIMARY KEY"),
                F.when(F.col("s_nationkey").isNull(), "NO NATION KEY PRESENT"),
                F.when(F.col("_valid_nation_key").isNull(), "INVALID NATION KEY"),
                F.when(F.col("s_phone").isNull() | (F.col("s_phone") == ""), "MISSING PHONE"),
            )
        )
    )
)

w = Window.partitionBy(pk_col).orderBy(F.col("_ingested_at").desc())

supplier_deduped = (
    supplier_validated
    .filter(F.size(F.col("quarantine_reason")) == 0)
    .withColumn(
        "quarantine_reason",
        F.when(F.row_number().over(w) > 1, "DUPLICATE PRIMARY KEY"),
    )
)

supplier_valid = (
    supplier_deduped
    .filter(F.col("quarantine_reason").isNull())
    .drop("quarantine_reason", "_valid_nation_key", "_ingested_at", "_source_file")
)

supplier_quarantined = (
    supplier_validated
    .filter(F.size(F.col("quarantine_reason")) > 0)
    .select(
        F.to_csv(F.struct(*supplier_std.columns)).alias("raw_data"),
        F.array_join(F.col("quarantine_reason"), ", ").alias("quarantine_reason"),
    )
    .unionByName(
        supplier_deduped
        .filter(F.col("quarantine_reason").isNotNull())
        .select(
            F.to_csv(F.struct(*supplier_std.columns)).alias("raw_data"),
            "quarantine_reason",
        )
    )
)

print(f"supplier: valid = {supplier_valid.count()}, quarantined = {supplier_quarantined.count()}")

supplier_valid.write.mode("overwrite").saveAsTable(f"{catalog}.silver.supplier")
supplier_quarantined.write.mode("overwrite").saveAsTable(f"{catalog}.silver.supplier_quarantine")

display(supplier_quarantined)

In [0]:
part_std = (
    spark.read.table(f"{catalog}.bronze.part")
    .select(
        "p_partkey",
        F.trim(F.col("p_name")).alias("p_name"),
        F.trim(F.col("p_mfgr")).alias("p_mfgr"),
        F.trim(F.col("p_brand")).alias("p_brand"),
        F.upper(F.trim(F.col("p_type"))).alias("p_type"),
        "p_size",
        F.upper(F.trim(F.col("p_container"))).alias("p_container"),
        "p_retailprice",
        F.trim(F.col("p_comment")).alias("p_comment"),
        "_ingested_at",
        "_source_file",
    )
)

pk_col = "p_partkey"

# Part don't have a foregin key
part_validated = (
    part_std
    .withColumn(
        "quarantine_reason",
        F.array_compact(
            F.array(
                F.when(F.col(pk_col).isNull(), "INVALID PRIMARY KEY"),
                F.when(F.col("p_size").isNull() | (F.col("p_size") < 0), "INVALID SIZE"),
                F.when(F.col("p_retailprice").isNull() | (F.col("p_retailprice") <= 0), "NON POSITIVE RETAIL PRICE"),
            )
        )
    )
)

w = Window.partitionBy(pk_col).orderBy(F.col("_ingested_at").desc())

part_deduped = (
    part_validated
    .filter(F.size(F.col("quarantine_reason")) == 0)
    .withColumn(
        "quarantine_reason",
        F.when(F.row_number().over(w) > 1, "DUPLICATE PRIMARY KEY"),
    )
)

part_valid = (
    part_deduped
    .filter(F.col("quarantine_reason").isNull())
    .drop("quarantine_reason", "_ingested_at", "_source_file")
)

part_quarantined = (
    part_validated
    .filter(F.size(F.col("quarantine_reason")) > 0)
    .select(
        F.to_csv(F.struct(*part_std.columns)).alias("raw_data"),
        F.array_join(F.col("quarantine_reason"), ", ").alias("quarantine_reason"),
    )
    .unionByName(
        part_deduped
        .filter(F.col("quarantine_reason").isNotNull())
        .select(
            F.to_csv(F.struct(*part_std.columns)).alias("raw_data"),
            "quarantine_reason",
        )
    )
)

print(f"part: valid = {part_valid.count()}, quarantined = {part_quarantined.count()}")

part_valid.write.mode("overwrite").saveAsTable(f"{catalog}.silver.part")
part_quarantined.write.mode("overwrite").saveAsTable(f"{catalog}.silver.part_quarantine")

display(part_quarantined)

In [0]:
partsupp_std = (
    spark.read.table(f"{catalog}.bronze.partsupp")
    .select(
        "ps_partkey",
        "ps_suppkey",
        "ps_availqty",
        "ps_supplycost",
        F.trim(F.col("ps_comment")).alias("ps_comment"),
        "_ingested_at",
        "_source_file",
    )
)

# composite primary key
pk_cols = ["ps_partkey", "ps_suppkey"]

valid_part_keys = spark.read.table(f"{catalog}.silver.part").select(
    F.col("p_partkey").alias("_valid_part_key")
)

valid_supplier_keys = spark.read.table(f"{catalog}.silver.supplier").select(
    F.col("s_suppkey").alias("_valid_supplier_key")
)

partsupp_validated = (
    partsupp_std
    .join(valid_part_keys, F.col("ps_partkey") == F.col("_valid_part_key"), how="left")
    .join(valid_supplier_keys, F.col("ps_suppkey") == F.col("_valid_supplier_key"), how="left")
    .withColumn(
        "quarantine_reason",
        F.array_compact(
            F.array(
                F.when(F.col("ps_partkey").isNull() | F.col("ps_suppkey").isNull(), "INVALID PRIMARY KEY"),
                F.when(F.col("ps_partkey").isNull(), "NO PART KEY PRESENT"),
                F.when(F.col("_valid_part_key").isNull(), "INVALID PART KEY"),
                F.when(F.col("ps_suppkey").isNull(), "NO SUPPLIER KEY PRESENT"),
                F.when(F.col("_valid_supplier_key").isNull(), "INVALID SUPPLIER KEY"),
                F.when(F.col("ps_availqty").isNull() | (F.col("ps_availqty") < 0), "INVALID AVAILQTY"),
                F.when(F.col("ps_supplycost").isNull() | (F.col("ps_supplycost") <= 0), "NON POSITIVE SUPPLYCOST"),
            )
        ),
    )
)

w = Window.partitionBy(*pk_cols).orderBy(F.col("_ingested_at").desc())

partsupp_deduped = (
    partsupp_validated
    .filter(F.size(F.col("quarantine_reason")) == 0)
    .withColumn(
        "quarantine_reason",
        F.when(F.row_number().over(w) > 1, "DUPLICATE PRIMARY KEY"),
    )
)

partsupp_valid = (
    partsupp_deduped
    .filter(F.col("quarantine_reason").isNull())
    .drop("quarantine_reason", "_valid_part_key", "_valid_supplier_key", "_ingested_at", "_source_file")
)

partsupp_quarantined = (
    partsupp_validated
    .filter(F.size(F.col("quarantine_reason")) > 0)
    .select(
        F.to_csv(F.struct(*partsupp_std.columns)).alias("raw_data"),
        F.array_join(F.col("quarantine_reason"), ", ").alias("quarantine_reason"),
    )
    .unionByName(
        partsupp_deduped
        .filter(F.col("quarantine_reason").isNotNull())
        .select(
            F.to_csv(F.struct(*partsupp_std.columns)).alias("raw_data"),
            "quarantine_reason",
        )
    )
)

print(f"partsupp: valid = {partsupp_valid.count()}, quarantined = {partsupp_quarantined.count()}")

partsupp_valid.write.mode("overwrite").saveAsTable(f"{catalog}.silver.partsupp")
partsupp_quarantined.write.mode("overwrite").saveAsTable(f"{catalog}.silver.partsupp_quarantine")

display(partsupp_quarantined)

In [0]:
lineitem_std = (
    spark.read.table(f"{catalog}.bronze.lineitem")
    .select(
        "l_orderkey",
        "l_partkey",
        "l_suppkey",
        "l_linenumber",
        "l_quantity",
        "l_extendedprice",
        "l_discount",
        "l_tax",
        F.upper(F.trim(F.col("l_returnflag"))).alias("l_returnflag"),
        F.upper(F.trim(F.col("l_linestatus"))).alias("l_linestatus"),
        "l_shipdate",
        "l_commitdate",
        "l_receiptdate",
        F.upper(F.trim(F.col("l_shipinstruct"))).alias("l_shipinstruct"),
        F.upper(F.trim(F.col("l_shipmode"))).alias("l_shipmode"),
        F.trim(F.col("l_comment")).alias("l_comment"),
        "_ingested_at",
        "_source_file",
    )
)

# composite primary key: (l_orderkey, l_linenumber)
pk_cols = ["l_orderkey", "l_linenumber"]

valid_order_keys = spark.read.table(f"{catalog}.silver.order").select(
    F.col("o_orderkey").alias("_valid_order_key")
)
valid_part_keys = spark.read.table(f"{catalog}.silver.part").select(
    F.col("p_partkey").alias("_valid_part_key")
)
valid_supplier_keys = spark.read.table(f"{catalog}.silver.supplier").select(
    F.col("s_suppkey").alias("_valid_supplier_key")
)

lineitem_validated = (
    lineitem_std
    .join(valid_order_keys, F.col("l_orderkey") == F.col("_valid_order_key"), how="left")
    .join(valid_part_keys, F.col("l_partkey") == F.col("_valid_part_key"), how="left")
    .join(valid_supplier_keys, F.col("l_suppkey") == F.col("_valid_supplier_key"), how="left")
    .withColumn(
        "quarantine_reason",
        F.array_compact(
            F.array(
                F.when(F.col("l_orderkey").isNull() | F.col("l_linenumber").isNull(), "INVALID PRIMARY KEY"),
                F.when(F.col("l_orderkey").isNull(), "NO ORDER KEY PRESENT"),
                F.when(F.col("_valid_order_key").isNull(), "INVALID ORDER KEY"),
                F.when(F.col("l_partkey").isNull(), "NO PART KEY PRESENT"),
                F.when(F.col("_valid_part_key").isNull(), "INVALID PART KEY"),
                F.when(F.col("l_suppkey").isNull(), "NO SUPPLIER KEY PRESENT"),
                F.when(F.col("_valid_supplier_key").isNull(), "INVALID SUPPLIER KEY"),
                F.when(F.col("l_quantity").isNull() | (F.col("l_quantity") <= 0), "INVALID QUANTITY"),
                F.when((F.col("l_extendedprice") <= 0), "NON POSITIVE EXTENDED PRICE"),
                F.when((F.col("l_discount") < 0) | (F.col("l_discount") > 0.10), "INVALID DISCOUNT"),
                F.when(F.col("l_tax").isNull() | (F.col("l_tax") < 0) | (F.col("l_tax") > 0.08), "INVALID TAX"),
                
                F.when(
                    F.col("l_shipdate").isNull() | F.col("l_commitdate").isNull() | F.col("l_receiptdate").isNull(),
                    "MISSING DATE",
                ),
                F.when(F.col("l_shipdate") > F.col("l_receiptdate"), "SHIP DATE AFTER RECEIPT DATE"),
                F.when((F.col("l_shipdate") < "1992-01-01") | (F.col("l_shipdate") > "1998-12-31"), "SHIP DATE OUT OF RANGE"),
                F.when((F.col("l_commitdate") < "1992-01-01") | (F.col("l_commitdate") > "1998-12-31"), "COMMIT DATE OUT OF RANGE"),
                F.when((F.col("l_receiptdate") < "1992-01-01") | (F.col("l_receiptdate") > "1998-12-31"), "RECEIPT DATE OUT OF RANGE"),
            )
        ),
    )
)

w = Window.partitionBy(*pk_cols).orderBy(F.col("_ingested_at").desc())

lineitem_deduped = (
    lineitem_validated
    .filter(F.size(F.col("quarantine_reason")) == 0)
    .withColumn(
        "quarantine_reason",
        F.when(F.row_number().over(w) > 1, "DUPLICATE PRIMARY KEY"),
    )
)

lineitem_valid = (
    lineitem_deduped
    .filter(F.col("quarantine_reason").isNull())
    .drop("quarantine_reason", "_valid_order_key", "_valid_part_key", "_valid_supplier_key", "_ingested_at", "_source_file")
)

lineitem_quarantined = (
    lineitem_validated
    .filter(F.size(F.col("quarantine_reason")) > 0)
    .select(
        F.to_csv(F.struct(*lineitem_std.columns)).alias("raw_data"),
        F.array_join(F.col("quarantine_reason"), ", ").alias("quarantine_reason"),
    )
    .unionByName(
        lineitem_deduped
        .filter(F.col("quarantine_reason").isNotNull())
        .select(
            F.to_csv(F.struct(*lineitem_std.columns)).alias("raw_data"),
            "quarantine_reason",
        )
    )
)

print(f"lineitem: valid = {lineitem_valid.count()}, quarantined = {lineitem_quarantined.count()}")

lineitem_valid.write.mode("overwrite").saveAsTable(f"{catalog}.silver.lineitem")
lineitem_quarantined.write.mode("overwrite").saveAsTable(f"{catalog}.silver.lineitem_quarantine")

display(lineitem_quarantined)

In [0]:
from pyspark.sql import types as T

# (bronze table, silver table, rules applied in its cleaning cell)
dq_tables = [
    ("region", "region", ["missing_primary_key", "duplicate primary key"]),
    ("nation", "nation", ["missing_primary_key", "DUPLICATE PRIMARY KEY", "INVALID REGION KEY"]),
    ("customer", "customer", ["INVALID PRIMARY KEY", "NO NATION KEY PRESENT", "INVALID NATION KEY", "DUPLICAtE PRIMARY KEY"]),
    ("supplier", "supplier", ["INVALID PRIMARY KEY", "NO NATION KEY PRESENT", "INVALID NATION KEY", "MISSING PHONE", "DUPLICATE PRIMARY KEY"]),
    ("part", "part", ["INVALID PRIMARY KEY", "INVALID SIZE", "NON POSITIVE RETAIL PRICE", "DUPLICATE PRIMARY KEY"]),
    ("partsupp", "partsupp", ["INVALID PRIMARY KEY", "NO PART KEY PRESENT", "INVALID PART KEY", "NO SUPPLIER KEY PRESENT", "INVALID SUPPLIER KEY", "INVALID AVAILQTY", "NON POSITIVE SUPPLYCOST", "DUPLICATE PRIMARY KEY"]),
    ("orders", "order", ["INVALID PRIMARY KEY", "NO CUSTOMER KEY PRESENT", "INVALID NATION KEY", "NON POSITIVE TOTALPRICE", "ORDER DATE BEFORE 1992", "ORDER DATE AFTER 1998", "DUPLICAtE PRIMARY KEY"]),
    ("lineitem", "lineitem", ["INVALID PRIMARY KEY", "NO ORDER KEY PRESENT", "INVALID ORDER KEY", "NO PART KEY PRESENT", "INVALID PART KEY", "NO SUPPLIER KEY PRESENT", "INVALID SUPPLIER KEY", "INVALID QUANTITY", "NON POSITIVE EXTENDED PRICE", "INVALID DISCOUNT", "INVALID TAX", "MISSING DATE", "SHIP DATE AFTER RECEIPT DATE", "SHIP DATE OUT OF RANGE", "COMMIT DATE OUT OF RANGE", "RECEIPT DATE OUT OF RANGE", "DUPLICATE PRIMARY KEY"]),
]

report_rows = []

for bronze_name, silver_name, rules in dq_tables:
    bronze_rows = spark.table(f"{catalog}.bronze.{bronze_name}").count()
    valid_rows = spark.table(f"{catalog}.silver.{silver_name}").count()
    quarantined_rows = spark.table(f"{catalog}.silver.{silver_name}_quarantine").count()
    difference = bronze_rows - (valid_rows + quarantined_rows)

    # actual failures per rule (one quarantined row can carry several reasons)
    actual = {
        r["rule"]: r["count"]
        for r in (
            spark.table(f"{catalog}.silver.{silver_name}_quarantine")
            .select(F.explode(F.split(F.col("quarantine_reason"), ", ")).alias("rule"))
            .filter(F.length(F.trim("rule")) > 0)
            .groupBy("rule")
            .count()
            .collect()
        )
    }

    # every applied rule is reported, even when 0 records failed it
    for rule in rules + [r for r in actual if r not in rules]:
        report_rows.append(
            (
                bronze_name,
                silver_name,
                rule,
                int(actual.get(rule, 0)),
                f"Quarantined — record preserved in silver.{silver_name}_quarantine, excluded from silver.{silver_name}",
                int(bronze_rows),
                int(valid_rows),
                int(quarantined_rows),
            )
        )

    # reconciliation: valid + quarantined must equal the bronze count
    report_rows.append(
        (
            bronze_name,
            silver_name,
            "ROW COUNT RECONCILIATION (valid + quarantined vs bronze)",
            int(difference),
            "OK — no records lost" if difference == 0 else f"DATA LOSS — {difference} record(s) unaccounted, INVESTIGATE",
            int(bronze_rows),
            int(valid_rows),
            int(quarantined_rows),
        )
    )

schema = T.StructType(
    [
        T.StructField("bronze_table", T.StringType()),
        T.StructField("silver_table", T.StringType()),
        T.StructField("rule", T.StringType()),
        T.StructField("failing_records", T.LongType()),
        T.StructField("action_taken", T.StringType()),
        T.StructField("bronze_rows", T.LongType()),
        T.StructField("valid_rows", T.LongType()),
        T.StructField("quarantined_rows", T.LongType()),
    ]
)

dq_report = spark.createDataFrame(report_rows, schema).withColumn("generated_at", F.current_timestamp())

dq_report.write.mode("overwrite").saveAsTable(f"{catalog}.silver.data_quality_report")

losses = [r for r in report_rows if r[2].startswith("ROW COUNT") and r[3] != 0]
total_quarantined = sum(r[7] for r in report_rows if not r[2].startswith("ROW COUNT"))
print(f"DQ report: {len(dq_tables)} tables, {len(report_rows)} report rows, {total_quarantined} quarantined records total")
print("Reconciliation:", "ALL 8 TABLES OK — valid + quarantined = bronze" if not losses else losses)

display(dq_report.orderBy("bronze_table", "rule"))